In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/features.csv", parse_dates=["GAME_DATE"])
df = df.sort_values("GAME_DATE", kind="stable").reset_index(drop=True)

HOME_TEAM = "HOME_TEAM"
AWAY_TEAM = "AWAY_TEAM"

print(df.columns.tolist())
print(df[[HOME_TEAM, AWAY_TEAM]].head(3))

['GAME_ID', 'GAME_DATE', 'SEASON', 'HOME_TEAM_ID', 'HOME_TEAM', 'HOME_PTS', 'AWAY_TEAM_ID', 'AWAY_TEAM', 'AWAY_PTS', 'HOME_WIN', 'HOME_PTS_ROLL10', 'HOME_PTS_ALLOWED_ROLL10', 'HOME_WIN_ROLL10', 'HOME_REST_DAYS', 'AWAY_PTS_ROLL10', 'AWAY_PTS_ALLOWED_ROLL10', 'AWAY_WIN_ROLL10', 'AWAY_REST_DAYS']
  HOME_TEAM AWAY_TEAM
0       CHI       OKC
1       POR       MEM
2       SAC       HOU


In [5]:
def add_elo(df, k=20, home_adv=70, carry=0.75, gap_days=60):
    ratings = {}
    home_elo, away_elo = [], []
    prev_date = None

    for date, ht, at, hp, ap in zip(df["GAME_DATE"], df[HOME_TEAM], df[AWAY_TEAM],
                                    df["HOME_PTS"], df["AWAY_PTS"]):
        # new season: pull every rating 25% of the way back toward 1500
        if prev_date is not None and (date - prev_date).days > gap_days:
            for t in ratings:
                ratings[t] = 1500 + carry * (ratings[t] - 1500)
        prev_date = date

        h = ratings.get(ht, 1500)
        a = ratings.get(at, 1500)
        home_elo.append(h)
        away_elo.append(a)

        # expected home win probability, then update using the real result
        diff = h + home_adv - a
        expected = 1 / (1 + 10 ** (-diff / 400))
        margin = hp - ap
        home_won = 1 if margin > 0 else 0
        winner_diff = diff if home_won else -diff
        mov = (abs(margin) + 3) ** 0.8 / (7.5 + 0.006 * winner_diff)
        change = k * mov * (home_won - expected)
        ratings[ht] = h + change
        ratings[at] = a - change

    out = df.copy()
    out["HOME_ELO"] = home_elo
    out["AWAY_ELO"] = away_elo
    return out

df = add_elo(df)
print(df[["HOME_ELO", "AWAY_ELO"]].describe())

           HOME_ELO      AWAY_ELO
count  11896.000000  11896.000000
mean    1500.239057   1500.506554
std       99.644026     99.593826
min     1204.379498   1202.280007
25%     1438.040027   1436.949134
50%     1507.712436   1507.584267
75%     1570.479223   1570.894876
max     1798.143533   1801.813700


In [6]:
home_part = pd.DataFrame({"GAME_IDX": df.index, "GAME_DATE": df["GAME_DATE"],
                          "TEAM": df[HOME_TEAM], "IS_HOME": 1,
                          "NET": df["HOME_PTS"] - df["AWAY_PTS"]})
away_part = pd.DataFrame({"GAME_IDX": df.index, "GAME_DATE": df["GAME_DATE"],
                          "TEAM": df[AWAY_TEAM], "IS_HOME": 0,
                          "NET": df["AWAY_PTS"] - df["HOME_PTS"]})

long = pd.concat([home_part, away_part]).sort_values(["TEAM", "GAME_DATE"], kind="stable")

for w in (5, 20):
    long[f"NET_ROLL{w}"] = long.groupby("TEAM")["NET"].transform(
        lambda s: s.shift(1).rolling(w, min_periods=5).mean()
    )

for side, flag in (("HOME", 1), ("AWAY", 0)):
    part = long[long["IS_HOME"] == flag].set_index("GAME_IDX")[["NET_ROLL5", "NET_ROLL20"]]
    df = df.join(part.add_prefix(f"{side}_"))

In [7]:
df["ELO_DIFF"] = df["HOME_ELO"] - df["AWAY_ELO"]
df["NET5_DIFF"] = df["HOME_NET_ROLL5"] - df["AWAY_NET_ROLL5"]
df["NET20_DIFF"] = df["HOME_NET_ROLL20"] - df["AWAY_NET_ROLL20"]
df["NET10_DIFF"] = ((df["HOME_PTS_ROLL10"] - df["HOME_PTS_ALLOWED_ROLL10"])
                    - (df["AWAY_PTS_ROLL10"] - df["AWAY_PTS_ALLOWED_ROLL10"]))
df["REST_DIFF"] = df["HOME_REST_DAYS"] - df["AWAY_REST_DAYS"]

# manual leakage check: first game of a team must have no prior-form features
first = df.iloc[0]
print("First game, ELO:", first["HOME_ELO"], first["AWAY_ELO"], "(should be 1500, 1500)")
print("First game, NET_ROLL20:", first["HOME_NET_ROLL20"], "(should be NaN)")

df.to_csv("../data/processed/features_v2.csv", index=False)
print(df.shape)

First game, ELO: 1500.0 1500.0 (should be 1500, 1500)
First game, NET_ROLL20: nan (should be NaN)
(11896, 29)


In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

old_feats = ["HOME_PTS_ROLL10", "HOME_PTS_ALLOWED_ROLL10", "HOME_WIN_ROLL10", "HOME_REST_DAYS",
             "AWAY_PTS_ROLL10", "AWAY_PTS_ALLOWED_ROLL10", "AWAY_WIN_ROLL10", "AWAY_REST_DAYS"]
new_feats = ["ELO_DIFF", "NET5_DIFF", "NET20_DIFF"]
target_col = "HOME_WIN"

assert "HOME_PTS" not in old_feats + new_feats and "AWAY_PTS" not in old_feats + new_feats
assert target_col not in old_feats + new_feats

usable = df.dropna(subset=old_feats + new_feats)
cutoff = pd.Timestamp("2022-08-01")
train = usable[usable["GAME_DATE"] < cutoff]
test = usable[usable["GAME_DATE"] >= cutoff]

def evaluate(name, cols):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(train[cols], train[target_col])
    probs = model.predict_proba(test[cols])[:, 1]
    acc = accuracy_score(test[target_col], (probs >= 0.5).astype(int))
    ll = log_loss(test[target_col], probs)
    print(f"{name:<28} acc {acc:.4f}   log loss {ll:.4f}")

evaluate("Old 8 features", old_feats)
evaluate("Elo only", ["ELO_DIFF"])
evaluate("Old 8 + Elo", old_feats + ["ELO_DIFF"])
evaluate("Old 8 + Elo + net windows", old_feats + new_feats)

Old 8 features               acc 0.6345   log loss 0.6390
Elo only                     acc 0.6464   log loss 0.6230
Old 8 + Elo                  acc 0.6475   log loss 0.6193
Old 8 + Elo + net windows    acc 0.6461   log loss 0.6193
